# Unify EHR Data – Parse Clinical Notes and Load to Table

This notebook is the **unify_ehr_data** workflow. It:

1. Reads all 100 PDF clinical notes from the volume subfolder `clinical_notes`.
2. Parses each PDF with **ai_parse_document** (Agent Brick–style document parsing).
3. Writes the parsed results into **melissap.melissa_pang.parsed_clinical_notes**.

**Volume:** `/Volumes/melissap/melissa_pang/project_volume/clinical_notes/` (from the synthetic data notebook).

In [20]:
# Get Spark session (injected on Databricks; create via Databricks Connect when run locally)
try:
    spark
except NameError:
    try:
        from databricks.connect import DatabricksSession
        spark = DatabricksSession.builder.profile("DEFAULT").getOrCreate()
    except Exception:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
# Avoid sparkContext on Spark Connect (not supported)
print("Spark session ready.")

Spark session ready.


In [21]:
# Configuration
CATALOG = "melissap"
SCHEMA = "melissa_pang"
VOLUME_PATH = "/Volumes/melissap/melissa_pang/project_volume/clinical_notes"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.parsed_clinical_notes"

In [22]:
# Ensure catalog and schema exist and set context
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Table will be created as: {TABLE_NAME}")

I0000 00:00:1772735603.918310 11116377 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Table will be created as: melissap.melissa_pang.parsed_clinical_notes


In [23]:
# Read all PDFs from the volume as binary, parse with ai_parse_document, extract patient_id from path
# Uses Agent Brick–style document parsing (ai_parse_document) for the 100 clinical notes.
from pyspark.sql.functions import col, current_timestamp, expr, regexp_extract

df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(VOLUME_PATH)
)

parsed_df = (
    df
    .withColumn("parsed", expr("ai_parse_document(content)"))
    .withColumn("patient_id", regexp_extract(col("path"), r"patient_(PAT_\d+)", 1))
    .withColumn("parsed_at", current_timestamp())
    .select("path", "patient_id", "parsed", "parsed_at")
)

parsed_df.printSchema()
print(f"Parsed {parsed_df.count()} documents")

root
 |-- path: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- parsed: variant (nullable = true)
 |-- parsed_at: timestamp (nullable = false)

Parsed 100 documents


In [24]:
# Flatten parsed content to text for easier querying; keep parsed variant for full structure
from pyspark.sql.functions import expr

text_df = (
    parsed_df
    .withColumn(
        "parsed_text",
        expr("""
            concat_ws(
                '\\n\\n',
                transform(
                    try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                    element -> try_cast(element:content AS STRING)
                )
            )
        """)
    )
    .select("path", "patient_id", "parsed", "parsed_text", "parsed_at")
)

text_df.show(2, truncate=50)

+--------------------------------------------------+----------+--------------------------------------------------+--------------------------------------------------+-------------------------+
|                                              path|patient_id|                                            parsed|                                       parsed_text|                parsed_at|
+--------------------------------------------------+----------+--------------------------------------------------+--------------------------------------------------+-------------------------+
|dbfs:/Volumes/melissap/melissa_pang/project_vol...|   PAT_006|{"document":{"elements":[{"bbox":[{"coord":[50,...|Confidential - Synthetic Clinical Note\n\nCLINI...|2026-03-05 18:33:28.21698|
|dbfs:/Volumes/melissap/melissa_pang/project_vol...|   PAT_002|{"document":{"elements":[{"bbox":[{"coord":[50,...|Confidential - Synthetic Clinical Note\n\nCLINI...|2026-03-05 18:33:28.21698|
+---------------------------------------

In [25]:
# Parse structured fields from parsed_text (matches format from synthetic data notebook)
from pyspark.sql.functions import col, regexp_extract

structured_df = (
    text_df
    .withColumn("note_patient_id", regexp_extract(col("parsed_text"), r"Patient ID:\s*(\S+)", 1))
    .withColumn("note_date", regexp_extract(col("parsed_text"), r"Date:\s*([^\n]+)", 1))
    .withColumn("encounter", regexp_extract(col("parsed_text"), r"Encounter:\s*(\d+)", 1))
    .withColumn("chief_complaint", regexp_extract(col("parsed_text"), r"Chief Complaint:\s*([^\n]+)", 1))
    .withColumn("assessment", regexp_extract(col("parsed_text"), r"Assessment:\s*([^\n]+)", 1))
    .withColumn("plan", regexp_extract(col("parsed_text"), r"Plan:\s*([^\n]+)", 1))
    .withColumn("next_visit", regexp_extract(col("parsed_text"), r"Next visit:\s*([^\n]+)", 1))
)

structured_df.select(
    "patient_id", "note_patient_id", "note_date", "encounter",
    "chief_complaint", "assessment", "plan", "next_visit"
).show(3, truncate=30)

+----------+---------------+----------+---------+-----------------------------+------------------------------+------------------------------+----------+
|patient_id|note_patient_id| note_date|encounter|              chief_complaint|                    assessment|                          plan|next_visit|
+----------+---------------+----------+---------+-----------------------------+------------------------------+------------------------------+----------+
|   PAT_006|        PAT_006|2024-12-24|       24|   Routine oncology follow-up|New nodule noted; recommend...|Organization people informa...|2026-03-05|
|   PAT_002|        PAT_002|2025-01-27|       68|      Fatigue and weight loss|No interval change. Surveil...|Wrong figure perform partic...|2026-03-05|
|   PAT_017|        PAT_017|2025-07-27|       78|Cough and shortness of breath|Stable lung malignancy. Con...|Participant dream citizen n...|2026-03-05|
+----------+---------------+----------+---------+-----------------------------+---

In [26]:
# Write to Unity Catalog table (overwrite for full refresh)
# Table includes path, patient_id, parsed, parsed_text, parsed_at, plus note_patient_id, note_date, encounter, chief_complaint, assessment, plan, next_visit
structured_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(TABLE_NAME)
print(f"Table saved: {TABLE_NAME}")
spark.table(TABLE_NAME).count()

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 37ef5b8f-1f00-4d01-a268-7760025cef80).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- path: string (nullable = true)
-- patient_id: string (nullable = true)
-- parsed: variant (nullable = true)
-- parsed_text: string (nullable = true)
-- parsed_at: timestamp (nullable = true)


Data schema:
root
-- path: string (nullable = true)
-- patient_id: string (nullable = true)
-- parsed: variant (nullable = true)
-- parsed_text: string (nullable = true)
-- parsed_at: timestamp (nullable = true)
-- note_patient_id: string (nullable = true)
-- note_date: string (nullable = true)
-- encounter: string (nullable = true)
-

### Create the **unify_ehr_data** job in Databricks

To run this notebook as a job, mirror this project in the Databricks workspace using **Repos** (Git). Then the job can reference the notebook in the repo.

**See [DATABRICKS_REPO_SETUP.md](DATABRICKS_REPO_SETUP.md)** in the project root for:

1. Pushing this project to Git and creating a Databricks Repo that clones it  
2. Getting the notebook path in the workspace  
3. Setting `NOTEBOOK_IN_WORKSPACE_PATH` in the cell below and re-running it to create/update the job  

After you pull in Databricks, edits you make in Cursor and push to Git will be reflected when you pull in the repo.

In [ ]:
# Create or update the workflow job 'unify_ehr_data' (run after mirroring via Repos — see DATABRICKS_REPO_SETUP.md)
# Set to your notebook path in the workspace (no .ipynb). Example after Repo is created:
#   "/Workspace/Repos/your.email@domain/claude-playground/3.unify_ehr_data"
NOTEBOOK_IN_WORKSPACE_PATH = None

if NOTEBOOK_IN_WORKSPACE_PATH is None:
    print(
        "Job creation skipped. To create the job:\n"
        "  1. Mirror this project in Databricks using Repos — see DATABRICKS_REPO_SETUP.md.\n"
        "  2. Set NOTEBOOK_IN_WORKSPACE_PATH above to the notebook path (Copy path in Databricks).\n"
        "  3. Re-run this cell."
    )
else:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.jobs import Task, NotebookTask, Source

    w = WorkspaceClient(profile="DEFAULT")
    existing = list(w.jobs.list(name="unify_ehr_data", limit=5))
    tasks = [
        Task(
            task_key="parse_clinical_notes",
            description="Parse 100 PDF clinical notes and load to parsed_clinical_notes table",
            notebook_task=NotebookTask(
                notebook_path=NOTEBOOK_IN_WORKSPACE_PATH,
                source=Source.WORKSPACE,
            ),
        )
    ]
    if existing:
        job_id = existing[0].job_id
        w.jobs.reset(job_id=job_id, new_settings={"name": "unify_ehr_data", "tasks": [t.as_dict() for t in tasks]})
        print(f"Updated workflow job 'unify_ehr_data' (job_id={job_id})")
    else:
        job = w.jobs.create(name="unify_ehr_data", tasks=tasks)
        print(f"Created workflow job 'unify_ehr_data' (job_id={job.job_id})")

Created workflow job 'unify_ehr_data' (job_id=299500688250366)
